# Traffic Sign Detection with YOLOv8

This notebook demonstrates the process of training a YOLOv8 model for traffic sign detection using a custom dataset from Roboflow. The steps include:

1.  Setting up the environment and installing dependencies.
2.  Downloading and exploring the dataset.
3.  Training a YOLOv8s model.
4.  Evaluating the model's performance.
5.  Visualizing the results.

In [ ]:
# Première cellule
import tensorflow as tf
print("GPU disponible:", tf.test.gpu_device_name())

# Vérifier PyTorch
import torch
print("PyTorch GPU:", torch.cuda.is_available())

In [ ]:
import tensorflow as tf
import torch

print("🔍 Vérification GPU...")
print("TensorFlow GPU:", tf.test.gpu_device_name())
print("PyTorch GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("🎉 GPU activé! Device:", torch.cuda.get_device_name(0))
else:
    print("❌ Toujours pas de GPU - Essayons la solution 2")

In [ ]:
# CELLULE 1 - INSTALLATION TOUT EN UN
!pip install ultralytics roboflow albumentations opencv-python > /dev/null 2>&1
!pip install imagehash scikit-image Pillow matplotlib seaborn > /dev/null 2>&1
print("✅ Toutes les dépendances installées!")

This cell performs a quick test of the YOLOv8 model by loading a pre-trained nano model and running a basic inference on a sample image URL. This verifies that YOLO is working and can perform detections.

In [ ]:
# CELLULE 3 - TEST YOLO RAPIDE
from ultralytics import YOLO

print("🧪 Test YOLOv8...")
model = YOLO('yolov8n.pt')  # Modèle nano pour test rapide
print("✅ YOLOv8 chargé avec succès!")

# Test inference basique
results = model('https://ultralytics.com/images/bus.jpg', verbose=False)
print("🎉 Inference test réussi!")
print("🚀 Environnement PRÊT pour le vrai travail!")

This cell sets up Roboflow for dataset management. It requires a Roboflow API key to download a specified dataset (in this case, a traffic sign recognition dataset) in YOLOv8 format.

In [ ]:
# CELLULE 4 - SETUP ROBOFLOW
from roboflow import Roboflow

print("📥 Configuration Roboflow...")

# Tu devras créer un compte gratuit sur roboflow.com
# et obtenir ta clé API
rf = Roboflow(api_key="bTeU4UsgwcUn5j3dFGAV")

# Télécharger le premier dataset
project = rf.workspace("test-a9klp").project("traffic-sign-recognition-iwjxl")
dataset = project.version(1).download("yolov8")

print(f"✅ Dataset téléchargé: {dataset.location}")

In [ ]:
# CELLULE 5 - EXPLORATION DES DONNÉES
import os
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

# Chemin du dataset
dataset_path = "/content/traffic-sign-recognition-1"
print(f"📁 Dataset path: {dataset_path}")

# Explorer la structure
print("\n📂 Structure du dataset:")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Premier fichiers seulement
        if file.endswith(('.jpg', '.png', '.yaml', '.yml')):
            print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... et {len(files) - 5} autres fichiers')

# Lire le fichier de configuration
data_yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\n📊 Configuration YAML:")
print(f"Classes: {data_config['names']}")
print(f"Nombre de classes: {data_config['nc']}")
print(f"Dossiers train: {data_config['train']}")
print(f"Dossiers val: {data_config['val']}")

In [ ]:
# CELLULE 6 BIS - ANALYSE DÉTAILLÉE
print("🎯 ANALYSE DÉTAILLÉE DU DATASET:")
print(f"📍 Chemin: {dataset_path}")
print(f"🎯 Nombre de classes: {data_config['nc']}")
print(f"📚 Classes disponibles: {data_config['names']}")

# Compter les images par split
def count_files(dataset_path):
    splits = ['train', 'valid', 'test']
    counts = {}

    for split in splits:
        img_dir = os.path.join(dataset_path, split, 'images')
        if os.path.exists(img_dir):
            images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
            label_dir = os.path.join(dataset_path, split, 'labels')
            labels = [f for f in os.listdir(label_dir) if f.endswith('.txt')] if os.path.exists(label_dir) else []
            counts[split] = {'images': len(images), 'labels': len(labels)}

    return counts

counts = count_files(dataset_path)
print(f"\n📈 RÉPARTITION DES DONNÉES:")
for split, data in counts.items():
    print(f"  {split.upper()}: {data['images']} images, {data['labels']} labels")

total_images = sum(data['images'] for data in counts.values())
print(f"\n📊 TOTAL: {total_images} images annotées")

In [ ]:
print("🧹 NETTOYAGE MÉMOIRE AVANT ENTRAÎNEMENT...")

import gc
import torch

# Nettoyer la mémoire
gc.collect()
torch.cuda.empty_cache()

# Vérifier la mémoire après nettoyage
if torch.cuda.is_available():
    print(f"💾 Mémoire GPU après nettoyage: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print("✅ Mémoire nettoyée!")

In [ ]:
print("🔍 VÉRIFICATION ENVIRONNEMENT...")

import torch
from ultralytics import YOLO

print(f"🎯 PyTorch: {torch.__version__}")
print(f"🚀 CUDA: {torch.cuda.is_available()}")
print(f"💻 GPU: {torch.cuda.get_device_name(0)}")
print(f"💾 Mémoire GPU libre: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

print("✅ Environnement vérifié!")

# 📊 Dataset visualization
These cells preview images, show class distribution and basic dataset statistics before training.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import random
from collections import Counter
import os, random, math
from PIL import Image


def list_image_files(folder, exts={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}):
    p = Path(folder)
    if not p.exists():
        return []
    return [str(f) for f in p.rglob('*') if f.suffix.lower() in exts]



from pathlib import Path
candidates = {
    'train_dir': globals().get('train_dir') or globals().get('data_dir') or globals().get('dataset_path') or Path('data/train'),
    'labels_csv': globals().get('labels_csv') or Path('labels.csv'),
    'train_df': globals().get('train_df'),
    'class_names': globals().get('class_names')
}
print('Candidate dataset locations:')
for k,v in candidates.items():
    print(k, '->', v if v is not None else 'None')


if candidates['train_df'] is not None:
    df = candidates['train_df']
    if isinstance(df, (list, tuple)):
        df = pd.DataFrame(df)
    if 'label' in df.columns or 'class' in df.columns:
        label_col = 'label' if 'label' in df.columns else 'class'
        dist = df[label_col].value_counts()
        print('Found dataframe with', len(df), 'rows. Showing top classes:')
        display(dist.head(20))
        plt.figure(figsize=(8,4))
        dist.head(20).plot(kind='bar')
        plt.title('Top 20 class counts')
        plt.xlabel('Class')
        plt.ylabel('Count')
        plt.show()
    else:
        print('train_df found but no `label` or `class` column. Columns:', df.columns.tolist())
elif Path(candidates['labels_csv']).exists():
    try:
        df = pd.read_csv(candidates['labels_csv'])
        if 'label' in df.columns or 'class' in df.columns:
            col = 'label' if 'label' in df.columns else 'class'
            dist = df[col].value_counts()
            print('Loaded labels CSV with', len(df), 'rows.')
            display(dist.head(20))
            plt.figure(figsize=(8,4))
            dist.head(20).plot(kind='bar')
            plt.title('Top 20 class counts (from labels CSV)')
            plt.xlabel('Class')
            plt.ylabel('Count')
            plt.show()
        else:
            print('labels_csv loaded but no `label`/`class` column. Columns:', df.columns.tolist())
    except Exception as e:
        print('Failed to read labels CSV:', e)
else:
    # fallback: infer from folder names
    train_folder = Path(candidates['train_dir'])
    if train_folder.exists():
        subdirs = [d for d in train_folder.iterdir() if d.is_dir()]
        counts = {}
        for d in subdirs:
            counts[d.name] = len(list_image_files(d))
        if counts:
            import pandas as pd
            s = pd.Series(counts)
            print('Inferred class counts from folder structure:')
            display(s.sort_values(ascending=False).head(30))
            plt.figure(figsize=(10,4))
            s.sort_values(ascending=False).head(30).plot(kind='bar')
            plt.title('Top classes (by folder)')
            plt.xlabel('Class')
            plt.ylabel('Count')
            plt.show()
        else:
            print('No class subfolders found under', train_folder)

In [ ]:
# 1) Helper imports and functions for visualization
import os, random, math
from pathlib import Path
from collections import Counter
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image
except Exception as e:
    print('Make sure required packages are installed: pandas, matplotlib, pillow.\nError:', e)

def list_image_files(folder, exts={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}):
    p = Path(folder)
    if not p.exists():
        return []
    return [str(f) for f in p.rglob('*') if f.suffix.lower() in exts]

def show_sample_images(paths, cols=5, max_images=15, figsize=(12,6)):
    import matplotlib.pyplot as plt
    from PIL import Image
    n = min(len(paths), max_images)
    rows = math.ceil(n/cols)
    plt.figure(figsize=figsize)
    for i, p in enumerate(paths[:n]):
        try:
            img = Image.open(p).convert('RGB')
            plt.subplot(rows, cols, i+1)
            plt.imshow(img)
            plt.axis('off')
        except Exception as e:
            print('Error opening', p, e)
    plt.tight_layout()
    plt.show()

# 3) Show random sample images from training directory (if exists)
# Assumes 'candidates' dictionary is defined in a previous cell
if 'candidates' not in globals():
    print("Error: 'candidates' dictionary not found. Please run the previous cell.")
else:
    train_folder = Path(candidates['train_dir'])
    if train_folder.exists() and any(train_folder.iterdir()):
        # If structure is class subfolders, grab a few images per class
        subdirs = [d for d in train_folder.iterdir() if d.is_dir()]
        if subdirs:
            samples = []
            for d in subdirs:
                imgs = list(d.glob('*'))
                imgs = [str(i) for i in imgs if i.suffix.lower() in {'.jpg','.png','.jpeg'}]
                if imgs:
                    samples.extend(imgs[:3])
            if not samples:
                # fallback: list all images
                from glob import glob
                samples = list_image_files(train_folder)[:15]
        else:
            samples = list_image_files(train_folder)[:15]
        print(f'Showing {len(samples)} sample images from', train_folder)
        show_sample_images(samples, cols=5, max_images=15)
    else:
        print('Training folder not found or empty:', train_folder)
        print('If your dataset is in a different location, set train_dir = Path("/path/to/train") or provide train_df with image paths.')

This cell trains the YOLOv8 model on the downloaded traffic sign recognition dataset. It uses optimized parameters for lower RAM usage, such as a smaller model ('yolov8s.pt'), reduced image size (416px), smaller batch size (8), mixed precision, and no caching.

In [ ]:
print("🚀 LANCEMENT ENTRAÎNEMENT SÉCURISÉ...")

from ultralytics import YOLO

# Configuration SÉCURISÉE pour faible RAM
model = YOLO('yolov8s.pt')  # Modèle léger et stable

print("🎯 Configuration optimisée:")
print("   - Modèle: YOLOv8s (léger)")
print("   - Image size: 416px")
print("   - Batch: 8")
print("   - Mixed Precision: Activé")

results = model.train(
    data='/content/traffic-sign-recognition-1/data.yaml',
    epochs=80,              # Assez pour convergence
    imgsz=416,              # Taille réduite pour économiser RAM
    batch=8,                # Batch petit pour éviter crash mémoire
    patience=15,            # Arrêt précoce si stagnation
    device=0,               # GPU
    save=True,              # Sauvegarder checkpoints
    exist_ok=True,          # Écraser si existe
    val=True,               # Validation pendant entraînement
    cache=False,            # CRITIQUE: désactiver cache
    workers=2,              # Réduit pour économiser RAM
    amp=True,               # Mixed precision - économise mémoire
    optimizer='AdamW',      # Plus stable
    lr0=0.001,              # Learning rate modéré
    momentum=0.9,
    weight_decay=0.0005,
    warmup_epochs=2.0,
    box=7.5,
    cls=0.5,
    dfl=1.5,
)

print("✅ ENTRAÎNEMENT LANCÉ!")
print("⏱️  Temps estimé: 3-5 heures")
print("📈 Suis la progression ci-dessus")

This cell prepares a Python script (`test_model.py`) that will be used to test the trained model immediately after the training is complete. The script loads the best trained model and performs inference on a few validation images, displaying the results.

In [ ]:
print("🎯 PRÉPARATION TEST DU MODÈLE...")

def prepare_immediate_test():
    """Préparer le test dès que l'entraînement finit"""

    test_code = """
# À EXÉCUTER DÈS QUE L'ENTRAÎNEMENT TERMINE
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import glob

print("🚀 TEST DU MODÈLE ENTRAÎNÉ...")

# Trouver le meilleur modèle
model_path = "/content/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

print(f"✅ Modèle chargé: {model_path}")

# Test rapide sur une image de validation
val_images = glob.glob("/content/traffic-sign-recognition-1/valid/images/*.jpg")[:3]

for img_path in val_images:
    results = model(img_path)
    plotted = results[0].plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    plt.title(f"Détection: {img_path.split('/')[-1]}")
    plt.axis('off')
    plt.show()

    # Afficher les détections
    for r in results:
        for box in r.boxes:
            cls = int(box.cls)
            conf = float(box.conf)
            print(f"  {model.names[cls]}: {conf:.2f}")
"""

    with open('/content/test_model.py', 'w') as f:
        f.write(test_code)

    print("✅ Script de test prêt!")
    print("💡 Exécute-le dès que l'entraînement termine")

prepare_immediate_test()

This cell executes the generated `test_model.py` script using the `%run` magic command. It loads the best trained YOLO model and performs object detection on a few sample validation images, displaying the results with bounding boxes and class labels.

In [ ]:
# 1. TESTER LE MODÈLE
%run /content/test_model.py

In [ ]:
# 2. ÉVALUER LES PERFORMANCES
from ultralytics import YOLO
model = YOLO('/content/runs/detect/train/weights/best.pt')
results = model.val()
print(f"🎯 PERFORMANCES FINALES:")
print(f"   mAP@0.5: {results.box.map50:.3f}")
print(f"   mAP@0.5:0.95: {results.box.map:.3f}")

print("\n   Précision par classe:")
for i, p in enumerate(results.box.p):
    print(f"     - {results.names[i]}: {p:.3f}")

print("\n   Rappel par classe:")
for i, r in enumerate(results.box.r):
    print(f"     - {results.names[i]}: {r:.3f}")

This cell generates and displays the learning curves from the training process. It plots the training and validation losses (box, classification, and DFL) and the precision, recall, and mAP metrics over the epochs to visualize how the model's performance changed during training.

In [ ]:
# CELLULE 12A - COURBES D'APPRENTISSAGE SEULEMENT
print("📈 GÉNÉRATION DES COURBES D'APPRENTISSAGE...")

from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd
import os

# Configuration pour figures compactes
plt.rcParams['figure.figsize'] = [10, 8]
plt.rcParams['font.size'] = 10

# Charger le modèle entraîné
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 1. COURBES D'APPRENTISSAGE
print("📊 Chargement des données d'entraînement...")
results_path = '/content/runs/detect/train/results.csv'

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)

    print("🎯 Génération des graphiques de loss et métriques...")

    # Graphique des losses
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Loss de boîtes
    axes[0,0].plot(results_df['epoch'], results_df['train/box_loss'], label='Train Box Loss', linewidth=1.5, color='blue')
    axes[0,0].plot(results_df['epoch'], results_df['val/box_loss'], label='Val Box Loss', linewidth=1.5, color='red')
    axes[0,0].set_title('Box Loss', fontsize=11, fontweight='bold')
    axes[0,0].set_xlabel('Epochs')
    axes[0,0].set_ylabel('Loss')
    axes[0,0].legend(fontsize=9)
    axes[0,0].grid(True, alpha=0.3)

    # Loss de classification
    axes[0,1].plot(results_df['epoch'], results_df['train/cls_loss'], label='Train Cls Loss', linewidth=1.5, color='blue')
    axes[0,1].plot(results_df['epoch'], results_df['val/cls_loss'], label='Val Cls Loss', linewidth=1.5, color='red')
    axes[0,1].set_title('Classification Loss', fontsize=11, fontweight='bold')
    axes[0,1].set_xlabel('Epochs')
    axes[0,1].set_ylabel('Loss')
    axes[0,1].legend(fontsize=9)
    axes[0,1].grid(True, alpha=0.3)

    # Métriques de précision
    axes[1,0].plot(results_df['epoch'], results_df['metrics/precision(B)'], label='Precision', linewidth=2, color='green')
    axes[1,0].plot(results_df['epoch'], results_df['metrics/recall(B)'], label='Recall', linewidth=2, color='orange')
    axes[1,0].set_title('Precision & Recall', fontsize=11, fontweight='bold')
    axes[1,0].set_xlabel('Epochs')
    axes[1,0].set_ylabel('Score')
    axes[1,0].legend(fontsize=9)
    axes[1,0].grid(True, alpha=0.3)
    axes[1,0].set_ylim(0, 1)

    # mAP
    axes[1,1].plot(results_df['epoch'], results_df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2, color='red')
    axes[1,1].plot(results_df['epoch'], results_df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2, color='purple')
    axes[1,1].set_title('mAP Metrics', fontsize=11, fontweight='bold')
    axes[1,1].set_xlabel('Epochs')
    axes[1,1].set_ylabel('mAP')
    axes[1,1].legend(fontsize=9)
    axes[1,1].grid(True, alpha=0.3)
    axes[1,1].set_ylim(0, 1)

    plt.suptitle('Courbes d\'Apprentissage - Modèle Panneaux Routiers', fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(pad=2.0)
    plt.savefig('/content/learning_curves_compact.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Afficher quelques statistiques clés
    final_epoch = results_df.iloc[-1]
    print(f"\n📊 STATISTIQUES FINALES D'APPRENTISSAGE:")
    print(f"   • Dernière epoch: {final_epoch['epoch']}")
    print(f"   • mAP@0.5 final: {final_epoch['metrics/mAP50(B)']:.3f}")
    print(f"   • mAP@0.5:0.95 final: {final_epoch['metrics/mAP50-95(B)']:.3f}")
    print(f"   • Précision finale: {final_epoch['metrics/precision(B)']:.3f}")
    print(f"   • Rappel final: {final_epoch['metrics/recall(B)']:.3f}")

else:
    print("❌ Fichier results.csv non trouvé")

print("✅ Courbes d'apprentissage générées avec succès!")

This cell generates and displays the confusion matrix and Precision-Recall (PR) curves for the trained model. The confusion matrix shows how well the model classified each class, and the PR curves illustrate the trade-off between precision and recall for different confidence thresholds for each class.

In [ ]:
# CELLULE 12B - MATRICE DE CONFUSION ET COURBES PR
print("🎯 GÉNÉRATION MATRICE DE CONFUSION ET COURBES PR...")

from PIL import Image
import matplotlib.pyplot as plt
import os

# Configuration pour figures compactes
plt.rcParams['figure.figsize'] = [10, 6]

# 2. MATRICE DE CONFUSION
print("🔍 Chargement de la matrice de confusion...")
confusion_matrix_path = '/content/runs/detect/train/confusion_matrix.png'

if os.path.exists(confusion_matrix_path):
    confusion_img = Image.open(confusion_matrix_path)

    # Afficher matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(confusion_img)
    ax.axis('off')
    ax.set_title('Matrice de Confusion - Détection des Panneaux', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix_compact.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Matrice de confusion affichée")
else:
    print("ℹ️ Génération de la matrice de confusion...")
    # Générer la matrice de confusion si elle n'existe pas
    model.val(save_dir='/content/runs/detect/confusion_analysis/', plots=True)

# 3. COURBES PRÉCISION-RAPPEL
print("📊 Chargement des courbes Precision-Rappel...")
pr_curve_path = '/content/runs/detect/train/PR_curve.png'

results = model.val(
    data='/content/traffic-sign-recognition-1/data.yaml',
    plots=True,
    save_json=True
)

if os.path.exists(pr_curve_path):
    pr_img = Image.open(pr_curve_path)

    # Afficher courbes PR
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(pr_img)
    ax.axis('off')
    ax.set_title('Courbes Precision-Rappel par Classe', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/pr_curve_compact.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Courbes Precision-Rappel affichées")
else:
    print("ℹ️ Courbes PR non trouvées")

# 4. RÉSUMÉ DES FICHIERS GÉNÉRÉS
print("\n📁 FICHIERS DE VISUALISATION GÉNÉRÉS:")
generated_files = [
    '/content/learning_curves_compact.png',
    '/content/confusion_matrix_compact.png',
    '/content/pr_curve_compact.png'
]

for file in generated_files:
    if os.path.exists(file):
        file_size = os.path.getsize(file) / 1024  # Taille en KB
        print(f"   ✅ {os.path.basename(file)} ({file_size:.1f} KB)")
    else:
        print(f"   ❌ {os.path.basename(file)} (non généré)")

print("🎉 Toutes les visualisations principales sont prêtes!")

This cell provides a compact analysis of the model's performance by displaying key global metrics (mAP@0.5, mAP@0.5:95, overall precision, and overall recall) in a formatted table and summary text.

In [ ]:
# CELLULE 13 MODIFIÉE - ANALYSE COMPACTE
print("🔍 ANALYSE DÉTAILLÉE COMPACTE...")

# Configuration pour tableau compact
plt.rcParams['figure.figsize'] = [8, 6]

# Ré-évaluer pour obtenir des données détaillées
results = model.val(plots=True, save_json=True)

# Afficher le rapport détaillé sous forme de tableau compact
print("\n📋 RAPPORT DÉTAILLÉ DES PERFORMANCES:")
print("="*45)

# Créer un tableau visuel compact
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('tight')
ax.axis('off')

# Prepare data for the table
table_data = []
headers = ['Metric', 'Score']

# Add global metrics to the table data
table_data.append(['mAP@0.5', f"{results.box.map50:.3f}"])
table_data.append(['mAP@0.5:95', f"{results.box.map:.3f}"])
table_data.append(['Overall Precision', f"{results.box.p.mean():.3f}"])
table_data.append(['Overall Recall', f"{results.box.r.mean():.3f}"])


# Créer le tableau
table = ax.table(cellText=table_data, colLabels=headers, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

plt.title('Performances Globales - Résumé Compact', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/performance_table_compact.png', dpi=150, bbox_inches='tight')
plt.show()

# Résumé statistique compact
print(f"\n📊 STATISTIQUES GLOBALES (Compact):")
print(f"   mAP@0.5:    {results.box.map50:.3f}")
print(f"   mAP@0.5:95: {results.box.map:.3f}")
print(f"   Précision:  {results.box.p.mean():.3f}")
print(f"   Rappel:     {results.box.r.mean():.3f}")

This cell performs and displays compact visual tests on a few sample validation images. It runs inference on a smaller number of images (4) and displays them in a 2x2 grid with detected objects and their confidence scores, providing a quick visual assessment of the model's performance.

In [ ]:
# CELLULE 14 MODIFIÉE - TESTS VISUELS COMPACTS
print("👁️ TESTS VISUELS COMPACTS...")

import glob
import cv2

# Configuration pour images plus petites
plt.rcParams['figure.figsize'] = [8, 5]

# Tester sur quelques images de validation
val_images = glob.glob('/content/traffic-sign-recognition-1/valid/images/*.jpg')[:4]  # 4 au lieu de 5

# Disposition 2x2 pour économiser de l'espace
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for i, (img_path, ax) in enumerate(zip(val_images, axes.flat)):
    # Prédiction
    results = model(img_path)

    # Affichage
    plotted = results[0].plot()
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{os.path.basename(img_path)[:15]}...', fontsize=10)
    ax.axis('off')

    # Ajouter les détections dans le titre
    detections = []
    for r in results:
        for box in r.boxes:
            cls = int(box.cls)
            conf = float(box.conf)
            detections.append(f"{model.names[cls]}:{conf:.2f}")

    if detections:
        ax.text(0.5, -0.1, ', '.join(detections[:2]),
               transform=ax.transAxes, ha='center', fontsize=8)

plt.tight_layout(pad=1.0)
plt.savefig('/content/test_predictions_compact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELLULE 15 - TEST INTERACTIF
print("🎮 TEST INTERACTIF AVEC TON MODÈLE!")

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Charger ton modèle super performant
model = YOLO('/content/runs/detect/train/weights/best.pt')

print("🚀 Modèle chargé! Testons sur quelques images...")

# Images de test intéressantes
test_images = [
    '/content/traffic-sign-recognition-1/valid/images/stop_1941_jpg.rf.3adadd69cd321acdd5fb32e6cdf55512.jpg',
    '/content/traffic-sign-recognition-1/valid/images/speed30_4967_jpg.rf.67f7ea8fbbf8cde5e79bea85a190b2f3.jpg',
    '/content/traffic-sign-recognition-1/valid/images/speed90_4367_jpg.rf.eef72f7ea3e0c9caf4f833d1c58d9b73.jpg'
]

for i, img_path in enumerate(test_images, 1):
    print(f"\n🔍 Test {i}/3: {img_path.split('/')[-1]}")

    # Faire la prédiction
    results = model(img_path)

    # Afficher le résultat
    plotted_image = results[0].plot()  # Image avec les détections
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(plotted_image, cv2.COLOR_BGR2RGB))
    plt.title(f'Détection - Test {i}', fontweight='bold')
    plt.axis('off')
    plt.show()

    # Montrer les détails
    print("📋 Détections trouvées:")
    for r in results:
        for box in r.boxes:
            class_id = int(box.cls)
            confidence = float(box.conf)
            class_name = model.names[class_id]
            print(f"   ✅ {class_name}: {confidence:.1%} de confiance")

print("\n🎉 TESTS TERMINÉS! Ton modèle fonctionne parfaitement!")

This cell saves the state dictionary of the best trained model to a file named `best_model.pt` and then provides a download link for this file, allowing the user to save the trained model weights locally.

In [ ]:
torch.save(model.state_dict(), "best_model.pt")
from google.colab import files
files.download("best_model.pt")

## Create a Gradio demo for the object detection model.

In [ ]:
%pip install gradio -q
print("Gradio installed successfully.")

### Create inference function

Define a Python function that takes an image as input, performs inference using the trained YOLO model, and returns the image with detections plotted.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def detect_objects(image):
    """
    Performs object detection on an input image using the trained YOLO model.

    Args:
        image: A numpy array representing the input image (RGB format).

    Returns:
        A numpy array representing the image with detections plotted.
    """
    # Load the trained YOLO model
    model = YOLO('/content/runs/detect/train/weights/best.pt')

    # Perform inference
    results = model(image)

    # Plot detections on the image
    plotted_image = results[0].plot() # results[0] is the Results object for the first image

    return plotted_image

print("`detect_objects` function defined.")

### Create and launch the Gradio interface


Create a Gradio interface using the `detect_objects` function and launch it.

In [ ]:
import gradio as gr
from PIL import Image

# Create the Gradio interface
# The inputs are defined as an image component
# The outputs are defined as an image component
# The `detect_objects` function is used to process the input image
iface = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type="numpy"),
    outputs=gr.Image(type="numpy"),
    title="Traffic Sign Detection with YOLOv8",
    description="Upload an image to detect traffic signs using a trained YOLOv8 model."
)

# Launch the interface
# The `share=True` option provides a public link for easy sharing
print("🚀 Launching Gradio interface...")
iface.launch(share=True)